In [16]:
import os
import copy
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [17]:
# Constants / path
data_dir = "image-results-spoofing/data"
train_dir = os.path.join(data_dir, "Train")
test_dir = os.path.join(data_dir, "Test")
batch_size = 32
learning_rate = 0.001
num_epochs = 200
RANDOM_STATE = 1082
criterion = nn.CrossEntropyLoss()

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set numpy random seed
np.random.seed(RANDOM_STATE)
# Set torch random seed
torch.manual_seed(RANDOM_STATE)
# If using CUDA
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [19]:
# Data transforms
train_transforms = transforms.Compose([
    transforms.Resize((64, 82)), # Resize to match the input size of photo
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
test_transforms = transforms.Compose([
    transforms.Resize((64, 82)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

In [20]:
# Datasets and DataLoaders
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

dataloaders = {'train': train_loader, 'test': test_loader}
classes = train_dataset.classes

In [21]:
# CNN Model Definition
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()
        self.layer1 = self.ConvModule(in_features=3, out_features=64)  # 16x16
        self.layer2 = self.ConvModule(in_features=64, out_features=128)  # 8x8
        self.layer3 = self.ConvModule(in_features=128, out_features=256)  # 4x4
        self.layer4 = self.ConvModule(in_features=256, out_features=512)  # 2x2
        self.gap = nn.AdaptiveAvgPool2d((2, 2))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2 * 2 * 512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x

    def ConvModule(self, in_features, out_features):
        return nn.Sequential(
            nn.Conv2d(in_channels=in_features, out_channels=out_features, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_features),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )


model = CNN(num_classes=len(classes)).to(device)
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)

In [22]:
def train_model(model, dataloaders, criterion, optimizer, num_epochs, device):
    since = time.time()
    history = {'train_loss': [], 'train_accuracy': [], 'test_loss': [], 'test_accuracy': []}

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        for phase in ['train', 'test']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += preds.eq(labels).sum().item()

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects / len(dataloaders[phase].dataset)

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.2%}')

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_accuracy'].append(epoch_acc)

            if phase == 'test' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best test Acc: {best_acc:.2%}')

    model.load_state_dict(best_model_wts)
    return model, history

# Train
model, history = train_model(model, dataloaders, criterion, optimizer, num_epochs, device)


Epoch 1/200
Train Loss: 0.4173, Acc: 85.03%
Test Loss: 0.3975, Acc: 86.31%

Epoch 2/200
Train Loss: 0.4008, Acc: 86.24%
Test Loss: 0.3930, Acc: 86.31%

Epoch 3/200
Train Loss: 0.3963, Acc: 86.24%
Test Loss: 0.3891, Acc: 86.31%

Epoch 4/200
Train Loss: 0.3943, Acc: 86.24%
Test Loss: 0.3847, Acc: 86.31%

Epoch 5/200
Train Loss: 0.3882, Acc: 86.24%
Test Loss: 0.3773, Acc: 86.31%

Epoch 6/200
Train Loss: 0.3824, Acc: 86.24%
Test Loss: 0.3643, Acc: 86.31%

Epoch 7/200
Train Loss: 0.3728, Acc: 86.24%
Test Loss: 0.3478, Acc: 86.31%

Epoch 8/200
Train Loss: 0.3632, Acc: 86.24%
Test Loss: 0.3316, Acc: 86.31%

Epoch 9/200
Train Loss: 0.3428, Acc: 86.44%
Test Loss: 0.3129, Acc: 86.31%

Epoch 10/200
Train Loss: 0.3082, Acc: 87.34%
Test Loss: 0.2453, Acc: 89.35%

Epoch 11/200
Train Loss: 0.2764, Acc: 89.41%
Test Loss: 0.4535, Acc: 86.45%

Epoch 12/200
Train Loss: 0.2448, Acc: 90.79%
Test Loss: 0.3341, Acc: 87.55%

Epoch 13/200
Train Loss: 0.1989, Acc: 92.83%
Test Loss: 0.1659, Acc: 94.05%

Epoch 14